In [0]:
from pyspark.sql import functions as F


def build_silver_weather(weather_bronze):
    weather_arrays = weather_bronze.select(
        "*",
        F.col("hourly.time").alias("_time"),
        F.col("hourly.temperature_2m").alias("_temperature"),
        F.col("hourly.precipitation").alias("_precipitation"),
        F.col("hourly.wind_speed_10m").alias("_wind_speed"),
    )

    array_checks = weather_arrays.select(
        "_source_file",
        F.size("_time").alias("time_count"),
        F.size("_temperature").alias("temperature_count"),
        F.size("_precipitation").alias("precipitation_count"),
        F.size("_wind_speed").alias("wind_speed_count"),
    )

    bad_arrays = array_checks.filter(
        (F.col("time_count") != F.col("temperature_count"))
        | (F.col("time_count") != F.col("precipitation_count"))
        | (F.col("time_count") != F.col("wind_speed_count"))
    )

    if bad_arrays.limit(1).count() > 0:
        raise ValueError(
            "Weather hourly arrays are misaligned."
        )

    weather_zipped = weather_arrays.withColumn(
        "_hourly_zipped",
        F.arrays_zip(
            "_time",
            "_temperature",
            "_precipitation",
            "_wind_speed",
        ),
    )

    weather_exploded = weather_zipped.select(
        "*",
        F.posexplode("_hourly_zipped").alias(
            "hour_index",
            "weather_hour",
        ),
    )

    silver_weather = weather_exploded.select(
        F.to_timestamp(
            F.col("weather_hour._time"),
            "yyyy-MM-dd'T'HH:mm",
        ).alias("weather_hour_local"),

        F.to_date(
            F.col("weather_hour._time")
        ).alias("weather_date_local"),

        F.col("weather_hour._temperature")
        .cast("double")
        .alias("temperature_2m_c"),

        F.col("weather_hour._precipitation")
        .cast("double")
        .alias("precipitation_mm"),

        F.col("weather_hour._wind_speed")
        .cast("double")
        .alias("wind_speed_10m_kmh"),

        F.col("timezone").alias("timezone"),

        F.col("_source_file").alias("source_file"),

        F.col("_ingested_at").alias("ingested_at"),
    )

    return silver_weather

In [0]:
BRONZE_WEATHER_TABLE = (
    "nyc_mobility.nyc_bronze.bronze_weather_raw"
)

SILVER_WEATHER_TABLE = (
    "nyc_mobility.nyc_silver.silver_weather_hourly"
)

In [0]:
weather_bronze = spark.table(
    BRONZE_WEATHER_TABLE
)

print("Bronze rows:", weather_bronze.count())

Bronze rows: 3


In [0]:
silver_weather = build_silver_weather(
    weather_bronze
)

In [0]:
silver_weather.printSchema()

display(
    silver_weather
    .orderBy("weather_hour_local")
    .limit(20)
)

root
 |-- weather_hour_local: timestamp (nullable = true)
 |-- weather_date_local: date (nullable = true)
 |-- temperature_2m_c: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- wind_speed_10m_kmh: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



weather_hour_local,weather_date_local,temperature_2m_c,precipitation_mm,wind_speed_10m_kmh,timezone,source_file,ingested_at
2026-03-01T00:00:00.000Z,2026-03-01,-2.6,0.0,0.4,America/New_York,/Volumes/nyc_mobility/nyc_bronze/nyc_source_files/landing/weather/weather_2026-03-01_2026-03-31.json,2026-09-15T19:46:46.205Z
2026-03-01T01:00:00.000Z,2026-03-01,-2.8,0.0,10.1,America/New_York,/Volumes/nyc_mobility/nyc_bronze/nyc_source_files/landing/weather/weather_2026-03-01_2026-03-31.json,2026-09-15T19:46:46.205Z
2026-03-01T02:00:00.000Z,2026-03-01,-2.0,0.0,5.0,America/New_York,/Volumes/nyc_mobility/nyc_bronze/nyc_source_files/landing/weather/weather_2026-03-01_2026-03-31.json,2026-09-15T19:46:46.205Z
2026-03-01T03:00:00.000Z,2026-03-01,-2.2,0.0,6.1,America/New_York,/Volumes/nyc_mobility/nyc_bronze/nyc_source_files/landing/weather/weather_2026-03-01_2026-03-31.json,2026-09-15T19:46:46.205Z
2026-03-01T04:00:00.000Z,2026-03-01,-2.2,0.0,7.7,America/New_York,/Volumes/nyc_mobility/nyc_bronze/nyc_source_files/landing/weather/weather_2026-03-01_2026-03-31.json,2026-09-15T19:46:46.205Z
2026-03-01T05:00:00.000Z,2026-03-01,-1.4,0.0,11.7,America/New_York,/Volumes/nyc_mobility/nyc_bronze/nyc_source_files/landing/weather/weather_2026-03-01_2026-03-31.json,2026-09-15T19:46:46.205Z
2026-03-01T06:00:00.000Z,2026-03-01,-0.2,0.0,15.0,America/New_York,/Volumes/nyc_mobility/nyc_bronze/nyc_source_files/landing/weather/weather_2026-03-01_2026-03-31.json,2026-09-15T19:46:46.205Z
2026-03-01T07:00:00.000Z,2026-03-01,-0.4,0.0,16.2,America/New_York,/Volumes/nyc_mobility/nyc_bronze/nyc_source_files/landing/weather/weather_2026-03-01_2026-03-31.json,2026-09-15T19:46:46.205Z
2026-03-01T08:00:00.000Z,2026-03-01,0.9,0.0,15.2,America/New_York,/Volumes/nyc_mobility/nyc_bronze/nyc_source_files/landing/weather/weather_2026-03-01_2026-03-31.json,2026-09-15T19:46:46.205Z
2026-03-01T09:00:00.000Z,2026-03-01,0.8,0.5,16.6,America/New_York,/Volumes/nyc_mobility/nyc_bronze/nyc_source_files/landing/weather/weather_2026-03-01_2026-03-31.json,2026-09-15T19:46:46.205Z


In [0]:
duplicate_hours = (
    silver_weather
    .groupBy("weather_hour_local")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_hours.count()

print("Duplicate weather hours:", duplicate_count)

if duplicate_count > 0:
    display(duplicate_hours)
    raise ValueError(
        "Duplicate weather hours detected."
    )

Duplicate weather hours: 0


In [0]:
required_nulls = silver_weather.agg(
    F.sum(
        F.col("weather_hour_local")
        .isNull()
        .cast("int")
    ).alias("weather_hour_nulls"),

    F.sum(
        F.col("timezone")
        .isNull()
        .cast("int")
    ).alias("timezone_nulls"),

    F.sum(
        F.col("source_file")
        .isNull()
        .cast("int")
    ).alias("source_file_nulls"),

    F.sum(
        F.col("ingested_at")
        .isNull()
        .cast("int")
    ).alias("ingested_at_nulls"),
)

display(required_nulls)

weather_hour_nulls,timezone_nulls,source_file_nulls,ingested_at_nulls
0,0,0,0


In [0]:
coverage = silver_weather.agg(
    F.min("weather_hour_local").alias("first_hour"),
    F.max("weather_hour_local").alias("last_hour"),
    F.count("*").alias("row_count"),
)

display(coverage)

first_hour,last_hour,row_count
2026-03-01T00:00:00.000Z,2026-05-31T23:00:00.000Z,2208


In [0]:
monthly_counts = (
    silver_weather
    .withColumn(
        "month",
        F.date_format(
            "weather_hour_local",
            "yyyy-MM",
        ),
    )
    .groupBy("month")
    .count()
    .orderBy("month")
)

display(monthly_counts)

month,count
2026-03,744
2026-04,720
2026-05,744


In [0]:
(
    silver_weather.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        SILVER_WEATHER_TABLE
    )
)